In [1]:
from langchain_ollama import ChatOllama
import pandas as pd
from sqlalchemy import text
from sqlalchemy import create_engine
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware
from langchain.agents.middleware import ModelCallLimitMiddleware

In [2]:
engine = create_engine("sqlite:///commerce.db")

In [3]:
llm = ChatOllama(
    model="qwen3.5:0.8b",
    temperature=0
)

response = llm.invoke("Hello! who are you ")

print(response.content)

Hello! I'm Qwen3.5, a large language model developed by Tongyi Lab. I can assist you with a wide range of tasks, from answering questions and writing creative content to analyzing data or solving complex problems. How can I help you today? 😊


In [4]:
@tool
def search_products(
    query: str = "",
    category: str = "",
    min_price: float = None,
    max_price: float = None,
    min_rating: float = None,
    min_stock: int = None,
    limit: int = 10
):
    """
    Search the product database.

    Use this tool whenever the customer asks about products,
    product categories, prices, ratings, availability, discounts,
    or popular products.

    query can be used to search product names.
    """
    
    sql = "SELECT * FROM products WHERE 1=1"
    params = {}

    if query:
        sql += " AND LOWER(product_name) LIKE LOWER(:query)"
        params["query"] = f"%{query}%"

    if category:
        sql += " AND LOWER(category) = LOWER(:category)"
        params["category"] = category

    if min_price is not None:
        sql += " AND price >= :min_price"
        params["min_price"] = min_price

    if max_price is not None:
        sql += " AND price <= :max_price"
        params["max_price"] = max_price

    if min_rating is not None:
        sql += " AND rating >= :min_rating"
        params["min_rating"] = min_rating

    if min_stock is not None:
        sql += " AND stock_quantity >= :min_stock"
        params["min_stock"] = min_stock

    sql += " ORDER BY rating DESC LIMIT :limit"
    params["limit"] = limit

    result = pd.read_sql(
        text(sql),
        engine,
        params=params
    )

    if result.empty:
        return "No matching products were found."

    return result.to_dict(orient="records")

In [5]:
tools = [search_products]

In [6]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are an ecommerce Product Agent.
    Your job is to help customers find products.
    Always use the search_products tool when answering
    questions about products.
    Never invent product information.
    Only provide product information returned by the tool.
    If no matching products are found, clearly tell the customer.
    Be concise, friendly, and helpful.
    """,
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 100),
            keep=("messages", 20),
            ),
        ModelCallLimitMiddleware(
            thread_limit=10,
            run_limit=5,
            exit_behavior="end",
        ),
    ],
    checkpointer=InMemorySaver(),
)

In [7]:
config = {
    "configurable": {
        "thread_id": "user_123"
    }
}

In [8]:
response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Show me products has rating 4"
        }
    ]
},
config=config                        
)

print(response["messages"][-1].content)

Here are 10 products with a rating of 4 or higher:

*   **Product 37** (Electronics) - $443.76 | Rating: 5.0 | Stock: 106
*   **Product 53** (Home & Kitchen) - $333.01 | Rating: 4.9 | Stock: 190
*   **Product 70** (Books) - $381.05 | Rating: 4.9 | Stock: 172
*   **Product 93** (Home & Kitchen) - $152.41 | Rating: 4.9 | Stock: 45
*   **Product 3** (Sports) - $494.66 | Rating: 4.8 | Stock: 32
*   **Product 26** (Clothing) - $174.48 | Rating: 4.8 | Stock: 31
*   **Product 54** (Toys) - $465.70 | Rating: 4.8 | Stock: 114
*   **Product 63** (Toys) - $198.82 | Rating: 4.8 | Stock: 59
*   **Product 64** (Beauty) - $301.59 | Rating: 4.8 | Stock: 157
*   **Product 91** (Books) - $297.63 | Rating: 4.8 | Stock: 181

Would you like more details about any of these products?


In [9]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Which one is the cheapest?"
            }
        ]
    },
    config=config
)

print(response["messages"][-1].content)

Based on the list of products with a rating of 4 or higher, here is the cheapest option:

**Product 93** (Home & Kitchen) - **$152.41** | Rating: 4.9 | Stock: 45

This product is significantly cheaper than the next most affordable item, Product 26 at $174.48.


In [10]:
def create_product_agent():

    agent = create_agent(
        model=llm,
        tools=[search_products],

        system_prompt="""You are an ecommerce Product Agent.
        Your job is to help customers find products.
        Always use the search_products tool when answering
        questions about products.
        Never invent product information.
        Only provide product information returned by the tool.
        If no matching products are found, clearly tell the customer.
        Be concise, friendly, and helpful.""",

        checkpointer=InMemorySaver()
    )

    return agent

In [11]:
product_agent = create_product_agent()

In [12]:
def chat_with_product_agent(user_id, message):

    config = {
        "configurable": {
            "thread_id": user_id
        }
    }

    response = product_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        },
        config=config
    )

    return response["messages"][-1].content

In [13]:
print(
    chat_with_product_agent(
        "user_123",
        "Show me  products."
    )
)

Here are 10 products available in our catalog:

1. **Product 37** - Electronics | $443.76 (20% off) | Stock: 106 | Rating: 5.0
2. **Product 53** - Home & Kitchen | $333.01 (15% off) | Stock: 190 | Rating: 4.9
3. **Product 70** - Books | $381.05 (15% off) | Stock: 172 | Rating: 4.9
4. **Product 93** - Home & Kitchen | $152.41 (30% off) | Stock: 45 | Rating: 4.9
5. **Product 3** - Sports | $494.66 (5% off) | Stock: 32 | Rating: 4.8
6. **Product 26** - Clothing | $174.48 (5% off) | Stock: 31 | Rating: 4.8
7. **Product 54** - Toys | $465.70 (25% off) | Stock: 114 | Rating: 4.8
8. **Product 63** - Toys | $198.82 (30% off) | Stock: 59 | Rating: 4.8
9. **Product 64** - Beauty | $301.59 (10% off) | Stock: 157 | Rating: 4.8
10. **Product 91** - Books | $297.63 (15% off) | Stock: 181 | Rating: 4.8

Would you like more details about any specific product or category?


In [14]:
print(
    chat_with_product_agent(
        "user_123",
        "Which one is the cheapest?"
    )
)

Based on the search results, here are the prices for each product:

1. **Product 93** - Home & Kitchen | $152.41 (30% off)
2. **Product 26** - Clothing | $174.48 (5% off)
3. **Product 63** - Toys | $198.82 (30% off)
4. **Product 1090** - Books | $297.63 (15% off)

The cheapest product is **Product 93**, which costs **$152.41**.


In [19]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.csv_loader import CSVLoader

C:\Users\youse\AppData\Local\Temp\ipykernel_27220\3447858497.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.csv_loader import CSVLoader


In [25]:
# Use the raw string (r"...") path to your CSV
csv_path = r"S:\Samir\Projects\telegram-commerce-agent\archive\Customer Support Data Set - Sheet1 (1).csv"

loader = CSVLoader(file_path=csv_path, encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} documents")

# Check the first one to make sure it looks right


Loaded 500 documents


In [26]:
print(documents[9].page_content)
print(documents[4].metadata)

question: I received the wrong item. What now?
answer: Please use our returns page to report the error, and we‚Äôll send the correct item immediately.
category: Returns
{'source': 'S:\\Samir\\Projects\\telegram-commerce-agent\\archive\\Customer Support Data Set - Sheet1 (1).csv', 'row': 4}


In [27]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

Created 500 chunks


In [28]:
for chunk in chunks[:3]:
    print("SOURCE:", chunk.metadata["source"])
    print(chunk.page_content)
    print("-" * 50)

SOURCE: S:\Samir\Projects\telegram-commerce-agent\archive\Customer Support Data Set - Sheet1 (1).csv
question: Where is my order?
answer: You can track your order using the tracking link sent to your email.
category: Shipping
--------------------------------------------------
SOURCE: S:\Samir\Projects\telegram-commerce-agent\archive\Customer Support Data Set - Sheet1 (1).csv
question: How do I return a shirt that doesn't fit?
answer: You can return any item within 30 days in its original condition using our return portal.
category: Returns
--------------------------------------------------
SOURCE: S:\Samir\Projects\telegram-commerce-agent\archive\Customer Support Data Set - Sheet1 (1).csv
question: My package hasn't arrived yet, what should I do?
answer: Please check the tracking number. If it hasn't updated in 3 days, contact our support team.
category: Shipping
--------------------------------------------------


In [29]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [30]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="customer_service"
)

In [31]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)